In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

def load_data():
    """
    Faz o download do dataset da internet (se não estiver em cache local)
    e retorna o DataFrame completo.
    """
    print("Carregando dataset California Housing...")
    # as_frame=True garante que retorne um DataFrame do Pandas
    data = fetch_california_housing(as_frame=True)
    df = data.frame
    return df

def feature_engineering(X):
    """
    Cria novas variáveis de negócio. Um cientista pleno não usa
    apenas as colunas brutas, ele cria relações lógicas.
    """
    X_new = X.copy()
    # Criando a proporção de quartos e quartos de dormir por residência
    X_new['Rooms_per_Household'] = X_new['AveRooms'] / X_new['HouseAge']
    X_new['Bedrooms_per_Room'] = X_new['AveBedrms'] / X_new['AveRooms']
    return X_new

def build_pipeline():
    """
    Constrói o pipeline modular garantindo que não haja data leakage.
    """
    # 1. Definir as colunas (todas são numéricas neste dataset)
    num_features = [
        'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
        'Population', 'AveOccup', 'Latitude', 'Longitude'
    ]

    # 2. Pipeline Numérico: Engenharia de Features Customizada + Escalonamento
    # Usamos o StandardScaler porque a Regressão Ridge é sensível à escala das variáveis
    num_transformer = Pipeline(steps=[
        ('feature_eng', FunctionTransformer(feature_engineering, validate=False)),
        ('scaler', StandardScaler())
    ])

    # 3. Empacotar no ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features)
        ])

    # 4. Pipeline Final com o Regressor Ridge
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', Ridge())
    ])

    return model_pipeline

def main():
    # 1. Ingestão de Dados
    df = load_data()

    # A variável alvo é o 'MedHouseVal' (Valor mediano da casa em centenas de milhares de dólares)
    X = df.drop('MedHouseVal', axis=1)
    y = df['MedHouseVal']

    # 2. Separação Treino/Teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 3. Construção e Otimização do Pipeline
    pipeline = build_pipeline()

    # Otimizando o hiperparâmetro alpha da Regressão Ridge
    param_grid = {
        'regressor__alpha': [0.1, 1.0, 10.0, 50.0, 100.0],
        'regressor__solver': ['auto', 'svd', 'cholesky']
    }

    print("\nIniciando treinamento e otimização (GridSearchCV)...")
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")

    # 4. Avaliação do Modelo
    y_pred = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("\n--- Resultados no Conjunto de Teste ---")
    # Multiplicando por 100.000 pois o dataset original está nessa escala
    print(f"Erro Médio Absoluto (MAE): US$ {mae * 100000:,.2f}")
    print(f"Raiz do Erro Quadrático Médio (RMSE): US$ {rmse * 100000:,.2f}")
    print(f"Coeficiente de Determinação ($R^2$): {r2:.4f}")

    # 5. Exportação
    joblib.dump(best_model, 'california_housing_model.pkl')
    print("\nModelo empacotado e salvo como 'california_housing_model.pkl'.")

if __name__ == "__main__":
    main()

Carregando dataset California Housing...

Iniciando treinamento e otimização (GridSearchCV)...
Melhores parâmetros encontrados: {'regressor__alpha': 50.0, 'regressor__solver': 'auto'}

--- Resultados no Conjunto de Teste ---
Erro Médio Absoluto (MAE): US$ 52,566.07
Raiz do Erro Quadrático Médio (RMSE): US$ 72,909.70
Coeficiente de Determinação ($R^2$): 0.5943

Modelo empacotado e salvo como 'california_housing_model.pkl'.
